# 赛马娘AI训练框架 - Colab一键训练

本笔记本用于在Google Colab上训练赛马娘育成AI。

## 步骤
1. 拉取代码
2. 安装依赖
3. 选择GPU
4. 准备数据
5. 开始训练或自我对弈
6. 推送模型到GitHub

In [ ]:
#@title 1. 环境检查
import torch
print(f'PyTorch版本: {torch.__version__}')
print(f'CUDA可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU内存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
#@title 2. 拉取代码
import os

# 设置GitHub仓库地址（替换为你的仓库）
GITHUB_REPO = 'your-username/uma-train' #@param {type:"string"}

if not os.path.exists('uma-train'):
    !git clone https://github.com/{GITHUB_REPO}.git
else:
    print('代码已存在，拉取最新版本...')
    %cd uma-train
    !git pull
    %cd ..

%cd uma-train

In [ ]:
#@title 3. 安装依赖
!pip install -r requirements.txt

In [ ]:
#@title 4. 验证框架
import sys
sys.path.insert(0, '.')

# 测试Game类
import random
from simulator import Game

rng = random.Random(42)
game = Game()
game.new_game(rng)
print('=== 游戏状态 ===')
print(game.print_state())
print(f'\n速度训练值: {game.train_value[0]}')
print(f'失败率: {game.fail_rate}')

# 测试模型
import torch
from model import create_model
from config import Game_Input_C

model = create_model('ems')
x = torch.randn(4, Game_Input_C)
output = model(x)
print(f'\n=== 模型测试 ===')
print(f'输入形状: {x.shape}')
print(f'输出形状: {output.shape}')
print(f'模型大小: {model.get_model_size_kb():.1f} KB')
print('框架验证通过!')

In [ ]:
#@title 5a. 自我对弈生成训练数据
import random
import numpy as np
from simulator import Game
from search import MCTS, SearchParam
from model.handwritten import HandwrittenEvaluator
from training.selfplay import SelfPlayWorker

NUM_GAMES = 10 #@param {type:"integer"}
SEARCH_SINGLE_MAX = 16 #@param {type:"integer"}
SEARCH_MAX_DEPTH = 3 #@param {type:"integer"}

evaluator = HandwrittenEvaluator()
search_param = SearchParam(
    search_single_max=SEARCH_SINGLE_MAX,
    max_depth=SEARCH_MAX_DEPTH,
)
worker = SelfPlayWorker(evaluator=evaluator, search_param=search_param)

rng = random.Random(42)
x, label = worker.generate_batch(NUM_GAMES, rng)

print(f'生成数据: x.shape={x.shape}, label.shape={label.shape}')

# 保存
os.makedirs('./data', exist_ok=True)
np.savez('./data/selfplay_train.npz', x=x, label=label)
print('数据已保存到 ./data/selfplay_train.npz')

In [ ]:
#@title 5b. 使用随机数据验证训练流程
from training.dataset import generate_random_data

# 生成随机数据
generate_random_data(10000, './data/train.npz')
generate_random_data(2000, './data/val.npz')

print('随机数据已生成')

In [ ]:
#@title 6. 开始训练
from training.train import train

# 训练参数
TRAIN_DATA = './data/train.npz' #@param {type:"string"}
VAL_DATA = './data/val.npz' #@param {type:"string"}
MODEL_TYPE = 'ems' #@param ['ems', 'tl', 'lin']
BATCH_SIZE = 256 #@param {type:"integer"}
MAX_STEP = 5000 #@param {type:"integer"}
LEARNING_RATE_SCALE = 1.0 #@param {type:"number"}
GPU_ID = 0 #@param {type:"integer"}

train(
    train_data_path=TRAIN_DATA,
    val_data_path=VAL_DATA,
    model_type=MODEL_TYPE,
    batch_size=BATCH_SIZE,
    max_step=MAX_STEP,
    lr_scale=LEARNING_RATE_SCALE,
    gpu=GPU_ID if torch.cuda.is_available() else -1,
)

In [ ]:
#@title 7. 推送模型到GitHub
import os

GITHUB_TOKEN = '' #@param {type:"string"}
COMMIT_MESSAGE = 'Update trained model' #@param {type:"string"}

if GITHUB_TOKEN:
    !git config user.email "colab@trainer.ai"
    !git config user.name "Colab Trainer"
    
    # 设置远程URL带token
    remote_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git'
    !git remote set-url origin {remote_url}
    
    !git add saved_models/
    !git commit -m "{COMMIT_MESSAGE}"
    !git push origin main
    print('模型已推送到GitHub!')
else:
    print('请填写GITHUB_TOKEN')

## 训练技巧

### 自我对弈+训练循环

```python
# AlphaZero风格：自我对弈 -> 训练 -> 自我对弈 -> ...
for generation in range(10):
    print(f'=== 第{generation}代 ===')
    # 1. 自我对弈
    x, label = worker.generate_batch(50, rng)
    np.savez(f'./data/sp_gen{generation}.npz', x=x, label=label)
    
    # 2. 训练
    train(f'./data/sp_gen{generation}.npz', max_step=2000)
```

### 超参数调整

- `search_single_max`: 搜索次数，越大越精确但越慢
- `max_depth`: 蒙特卡洛深度，推荐5-10
- `batch_size`: 根据GPU内存调整，T4推荐256-512
- `lr_scale`: 学习率缩放，初始训练用1.0

### 模型大小控制

模型大小需要控制在376KB以内（手机端推理限制）：
- `ems(1, 96, 2, 192, 192)` ≈ 300KB ✓
- `ems(1, 128, 3, 256, 256)` ≈ 700KB ✗